# hard15_run — model KHÁC LOẠI trên 15 câu khó (việc 1, kế hoạch 23/08)

**Câu hỏi đã viết ra trước:** ngoài cross-encoder bge-m3, có bộ chấm nào bắt được nhóm
câu mà nó chấm ~0 không? Đo trên **đúng 15 câu** và **đúng đầu vào** Gemini đã dùng
(`head` 260 ký tự + `excerpt` 900 ký tự, rổ 50 ứng viên) → so thẳng được với **Gemini 10/15**.

**Ngưỡng đặt TRƯỚC khi chạy** (số câu có gold trong top-5, trên 15 câu):

| ra | quyết định |
|---|---|
| **≥ 8/15** | hướng "model khác loại" là thật → đi tiếp (chấm cả dev300, rồi đề thi) |
| 4–7/15 | vùng mù, **KHÔNG mang lên đề thi**. Cần thêm bằng chứng trước |
| **≤ 3/15** | đóng hướng, 10/15 của Gemini là chuyện của riêng Gemini |

**Đối chứng bắt buộc `ce_ours`** — chính `AITeamVN/Vietnamese_Reranker` đang dùng, chấm
trên cùng `head+excerpt`. Pipeline thật chấm **đoạn sâu**, không phải head+excerpt, nên
đây là biến gây nhiễu chưa ai loại trừ:

- `ce_ours` ra **0–2/15** → khác biệt nằm ở **model**. Đúng giả thuyết, đi tiếp.
- `ce_ours` ra **≥ 5/15** → khác biệt nằm ở **ĐẦU VÀO** (head+excerpt tốt hơn đoạn sâu),
  không phải ở model. Việc phải làm khi đó là đổi cách dựng đoạn, **rẻ hơn nhiều** —
  và mọi kết luận "cần model khác loại" phải viết lại.

**Chi phí:** 750 cặp/model, 4 model ≈ **15–25 phút GPU**. T4 hoặc P100 đều được.

> 🔴 **BẬT GPU TRƯỚC KHI CHẠY.** Settings (cột phải) → Accelerator → **GPU T4 x2** hoặc
> **P100**. Session CPU của Kaggle dùng bản torch không có CUDA: nó tải xong vài GB trọng
> số rồi mới chết với `Torch not compiled with CUDA enabled`. Bước 1 đã có assert chặn
> việc đó trong 1 giây — nếu nó dừng ngay ở dòng đầu thì đúng là quên bật accelerator.

**Phải upload lên dataset `project-ir`:** `rerank_qwen.py` (BẢN MỚI 23/08, đè bản cũ) ·
`rerank.py` · `batch_01.json` `batch_02.json` `batch_03.json` (từ `Ketqua_E/gemini/`) ·
`dev_300_locked.json` (đã có). **Internet ON** để tải model.

> ⚠️ `batch_04..10.json` là RÁC của lượt dựng cũ (150 câu dễ), **không** phải câu khó.
> Bước 1 lọc theo danh sách 15 qid cứng nên có lỡ upload nhầm cũng không sai kết quả.

> ⚠️ **Giới hạn của phép đo này:** chấm **theo từng cặp** (pointwise, P("yes")), còn
> Gemini xếp hạng **cả 50 cùng lúc** và có suy luận. Nếu ra ≤3/15 thì chưa chắc đã bác
> được "model khác loại" — có thể chỉ bác được "pointwise". Muốn tách hai cái đó thì
> chạy lại đúng 15 câu này ở chế độ sinh văn bản xếp hạng cả rổ. Ghi vào CLAUDE.md
> đúng như vậy, đừng ghi gọn thành "model khác loại thua".


In [ ]:
!pip install -q -U "transformers>=4.51" bitsandbytes accelerate sentence-transformers

In [ ]:
# ===== Bước 1: cấu hình + nạp 15 câu khó + dựng 750 cặp =====
import os, sys, json, time, glob, hashlib, gc
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Chặn TRƯỚC khi tải model: session CPU sẽ tải xong ~8GB trọng số rồi mới chết ở .to("cuda")
# với "Torch not compiled with CUDA enabled" — 3 phút cho một lỗi phát hiện được trong 1 giây.
import torch
assert torch.cuda.is_available(), (
    "KHÔNG CÓ GPU. Settings (cột phải) -> Accelerator -> GPU T4 x2 hoặc P100 -> chạy lại. "
    f"[torch {torch.__version__} · bản CUDA = {torch.version.cuda}]")
print(f"GPU: {torch.cuda.get_device_name(0)} · torch {torch.__version__} · cuda {torch.version.cuda}")

DOC_CHARS = 1300          # head 260 + excerpt 900 + xuống dòng = 1161 -> không bao giờ cắt
MAX_LEN   = 1536          # ~530 token/cặp. Chạm trần là raise, không âm thầm cắt đuôi
GEMINI_HIT = 10           # mốc so: Gemini 2.5 Flash bắt 10/15 trên ĐÚNG bộ này
DI_TIEP, DONG = 8, 3

# 15 câu khó, chép cứng từ out_hard_01.md. Danh sách này là phép kiểm: upload nhầm lô
# nào cũng không lọt vào được, mà thiếu câu nào là assert bên dưới báo ngay.
HARD15 = set("112344 119400 128326 138214 1524 156454 159444 16942 33938 37542 "
             "49070 52126 65498 8946 92466".split())

INPUT_DIR = next(p for p in ("/kaggle/input/project-ir",
                             "/kaggle/input/datasets/locdovan211/project-ir")
                 if os.path.isdir(p))
OUT = "/kaggle/working/outputs"; os.makedirs(OUT, exist_ok=True)
sys.path.append(INPUT_DIR)

for f in ("rerank_qwen.py", "rerank.py"):
    b = open(f"{INPUT_DIR}/{f}", "rb").read()
    print(f"{f:16} {len(b):>6} bytes  {hashlib.sha256(b).hexdigest()[:12]}")
import rerank_qwen as RQ
assert "instruct" in RQ._TPL, "rerank_qwen.py là BẢN CŨ (chưa có _TPL) — upload lại"
from rerank import load_reranker

# gom mọi batch_*.json ở bất kỳ đâu trong dataset, LỌC theo HARD15
items = {}
for p in glob.glob(f"{INPUT_DIR}/**/batch_*.json", recursive=True):
    for it in json.load(open(p, encoding="utf-8")):
        if it["qid"] in HARD15:
            items[it["qid"]] = it
assert set(items) == HARD15, f"thiếu {sorted(HARD15 - set(items))} — upload batch_01..03.json"

dev = json.load(open(f"{INPUT_DIR}/dev_300_locked.json", encoding="utf-8"))
GOLD = {q: {str(a) for a in dev[q]["answer"]} for q in HARD15}

qids  = sorted(items)
pairs, index = [], []                       # index[i] = (qid, doc_id) của pairs[i]
for q in qids:
    it = items[q]
    for c in it["candidates"]:
        pairs.append([it["question"], (c["head"] + "\n" + c["excerpt"])[:DOC_CHARS]])
        index.append((q, str(c["doc_id"])))

assert len(pairs) == 750, len(pairs)
tran = sum(1 for q in qids if GOLD[q] & {d for qq, d in index if qq == q})
print(f"\n{len(qids)} câu · {len(pairs)} cặp · dài nhất {max(len(d) for _, d in pairs)} ký tự")
print(f"TRẦN: gold còn trong rổ ở {tran}/15 câu — không model nào vượt được")
print(f"Mốc: Gemini {GEMINI_HIT}/15 · bài mình 0/15 (15 câu này được chọn VÌ mình sai)")

# SÀN: đếm từ trùng thuần, 0 giây GPU. Ngưỡng đóng ≤3 đặt đúng ở đây — model nào
# không hơn được phép đếm từ thì không mang gì mới vào cả.
import re as _re
_t = lambda s: set(_re.findall(r"\w+", s.lower()))
_s = {}
for (q, d), p in zip(index, pairs):
    _s.setdefault(q, {})[d] = len(_t(p[0]) & _t(p[1]))
SAN = sum(1 for q in qids if GOLD[q] & set(sorted(_s[q], key=lambda d: -_s[q][d])[:5]))
print(f"SÀN: đếm từ trùng thuần = {SAN}/15  <- ngưỡng ĐÓNG ({DONG}) đặt ở đây, không tuỳ tiện")

In [ ]:
# ===== Bước 2: chấm — mỗi model một vòng, hỏng model này không giết model kia =====
import torch

MODELS = [
    # tag,        model_id,                        kwargs cho load_reranker
    ("ce_ours",   "AITeamVN/Vietnamese_Reranker",  {}),                                  # ĐỐI CHỨNG
    ("qwen3rr4b", "Qwen/Qwen3-Reranker-4B",        dict(load_4bit=True, batch_size=8)),
    ("qwen3rr8b", "Qwen/Qwen3-Reranker-8B",        dict(load_4bit=True, batch_size=4)),
    ("qwen25i7b", "Qwen/Qwen2.5-7B-Instruct",      dict(load_4bit=True, batch_size=8,
                                                        template="instruct")),
]

def chot(sc):
    """750 điểm phẳng -> {qid: [top-5 doc_id]} + {qid: {doc_id: điểm}}."""
    per = {}
    for (q, d), s in zip(index, sc):
        per.setdefault(q, {})[d] = float(s)
    top = {q: sorted(v, key=lambda d: -v[d])[:5] for q, v in per.items()}
    return per, top

res = {}
for tag, mid, kw in MODELS:
    try:
        t0 = time.time()
        m = load_reranker(mid, device="cuda", max_length=MAX_LEN, **kw)
        per, top = chot(m.predict(pairs))
        el = time.time() - t0

        hit  = sum(1 for q in qids if GOLD[q] & set(top[q]))
        top1 = sum(1 for q in qids if top[q][0] in GOLD[q])
        res[tag] = (hit, top1, el)

        json.dump(per, open(f"{OUT}/scores_hard15_{tag}.json", "w", encoding="utf-8"),
                  ensure_ascii=False)
        with open(f"{OUT}/out_hard_{tag}.md", "w", encoding="utf-8") as f:
            f.write("| qid | answer |\n| --- | --- |\n")
            for q in qids:
                f.write(f"| {q} | {', '.join(top[q])} |\n")
        print(f"\n>>> {tag}: {hit}/15 câu bắt được · {top1}/15 xếp hạng 1 · {el/60:.1f} phút")
        print(f"    bắt được: {[q for q in qids if GOLD[q] & set(top[q])]}")
    except Exception as e:
        print(f"\n>>> {tag}: HỎNG — {type(e).__name__}: {str(e)[:300]}")
        res[tag] = None
    finally:
        for v in ("m", "per", "top"):
            globals().pop(v, None)
        gc.collect(); torch.cuda.empty_cache()

print(f"\nĐÃ LƯU {OUT}/ — TẢI VỀ TRƯỚC KHI ĐÓNG PHIÊN (quy tắc 2)")

In [ ]:
# ===== Bước 3: đọc kết quả theo ngưỡng đã đặt trước =====
print(f"{'model':12s} {'bắt/15':>7s} {'hạng1/15':>9s} {'phút':>6s}")
print(f"{'Gemini 2.5':12s} {GEMINI_HIT:>7d} {'—':>9s} {'—':>6s}   (mốc)")
for tag, _, _ in MODELS:
    r = res.get(tag)
    print(f"{tag:12s} {'HỎNG':>7s}" if r is None else
          f"{tag:12s} {r[0]:>7d} {r[1]:>9d} {r[2]/60:>6.1f}")

ctl = res.get("ce_ours")
best = max(((t, r[0]) for t, r in res.items() if r and t != "ce_ours"),
           key=lambda x: x[1], default=(None, -1))
print("\n" + "=" * 64)
if ctl and ctl[0] >= 5:
    print(f"!! ĐỐI CHỨNG ce_ours = {ctl[0]}/15 >= 5 -> khác biệt là do ĐẦU VÀO, không phải model.")
    print("   Đổi hướng: sửa cách dựng đoạn (head+excerpt) chứ đừng đi tìm model mới.")
    print("   Mọi con số dưới đây phải đọc TRỪ ĐI mốc này, không phải trừ 0.")
elif ctl:
    print(f"đối chứng ce_ours = {ctl[0]}/15 -> đầu vào không phải nguyên nhân. Đọc tiếp:")

t, h = best
if t is None:      print("Không model nào chạy được. Xem lỗi ở Bước 2.")
elif h >= DI_TIEP: print(f"{t} = {h}/15 >= {DI_TIEP} -> ĐI TIẾP: chấm cả dev300 bằng model này.")
elif h <= DONG:    print(f"{t} = {h}/15 <= {DONG} -> ĐÓNG hướng pointwise. Xem cảnh báo ở đầu"
                         f" notebook trước khi ghi 'model khác loại thua' vào CLAUDE.md.")
else:              print(f"{t} = {h}/15 -> VÙNG MÙ. KHÔNG mang lên đề thi. 15 câu thì mỗi câu"
                         f" đáng 6,7%% — chênh dưới 3 câu là chưa đọc được.")
print("=" * 64)